# Landsat 지표면온도(LST) 정제 파이프라인 — 산단 열섬 측정

**목적**: 위성 원본(GeoTIFF)에서 산업단지 주변 지표면온도 차이(ΔLST)를 뽑고, 이를 기온 차이로 환산하는 전 과정을 **처음부터 끝까지 재현 가능하게** 담는다.

이 노트북은 `산업전력_검증노트북_2026-07-18.ipynb`의 §14·§15를 **독립 실행 가능하도록 추출**한 것이다. 원 노트북은 전력·온열질환 분석까지 포함하지만, 여기서는 위성 처리만 다루므로 **전력 자료 없이도 돌아간다.**

---

## 0. 필요한 것

### 파이썬 패키지
```
pandas · numpy · scipy · statsmodels · geopandas · pyogrio · shapely · pyproj · rasterio
```
`rasterio`가 핵심이다(GeoTIFF 읽기). conda 환경 권장.

### 자료 (모두 무료·공개)

| 폴더/파일 | 내용 | 출처 |
|---|---|---|
| `LANDSAT/` | Landsat 8/9 Collection 2 **Level-2** 여름 장면 | [USGS EarthExplorer](https://earthexplorer.usgs.gov) |
| `0718_DAM_PDAN/` | 산업단지 대표점 (EPSG:5186) | [산업입지정보시스템 ILIS](https://www.industryland.or.kr) |
| `0718_DAM_YUCH/` | 산업단지 유치업종 폴리곤 | ILIS |
| `OPEN_0720_기상청_{ASOS,AWS}지점정보.csv` | 관측소 좌표 (§6에서만 필요) | [기상자료개방포털](https://data.kma.go.kr) |
| `OPEN_*_기상청_{ASOS,AWS}일자료*.csv` | 관측소 일별 기온 (§6) | 기상자료개방포털 |

### Landsat 받는 법 — 밴드 선택이 중요하다

EarthExplorer → `Landsat` → `Landsat Collection 2 Level-2` → **`Landsat 8-9 OLI/TIRS C2 L2`** 로 검색한 뒤, Bulk Download에서 **`Options` → `File Groups` → *Level-2 Surface Temperature Bands*** 를 열어 다음 **4개만** 체크한다.

```
ST_B10.TIF     지표면온도 (본체)
QA_PIXEL.TIF   품질 비트 — 구름·그림자·바다 판별
ST_QA.TIF      화소별 불확실도
MTL.txt        촬영시각·구름량 메타데이터
```

- 전체 Product Bundle을 받으면 장면당 **~800MB**, 4개만 받으면 **~80MB**로 준다.
- `ST_B6.TIF`는 Landsat 4/5/7용 열밴드라 **불필요**하다.
- Surface Reflectance 그룹(`SR_B*`)도 이 분석엔 **불필요**하다. 식생지수(NDVI)·녹지 분석을 추가할 때만 `SR_B4`·`SR_B5`가 필요하다.
- 한반도 8개 산단 + 개별 폐공장 부지를 모두 덮으려면 WRS-2 **Path 114~116 / Row 034~036** 이 필요하다. §2의 커버리지 진단이 어느 부지가 왜 비었는지 알려준다.

> ⚠ 원본 위성영상은 저장소에 포함하지 않는다(101장 = 13.7GB). 위 절차로 각자 받는다.

---
## 1. 환경 + 변환 상수

`ST_B10`은 정수(DN)로 저장돼 있어 실제 온도로 바꿔야 한다. 변환식은 USGS **Data Format Control Book**에 명시돼 있다.

```
켈빈 = DN × 0.00341802 + 149.0
섭씨 = 켈빈 − 273.15
불확실도(K) = ST_QA × 0.01
```

`QA_PIXEL`은 16비트 플래그다. 비트별 의미:

| 비트 | 의미 | 처리 |
|---|---|---|
| 0 | fill (촬영범위 밖) | 제외 |
| 1 | dilated cloud | 제외 |
| 2 | cirrus (권운) | 제외 |
| 3 | cloud | 제외 |
| 4 | cloud shadow | 제외 |
| 5 | snow | 제외 |
| 6 | clear | — |
| **7** | **water** | **제외 ★** |

**bit 7(바다) 제외가 특히 중요하다.** 대상 산단 11곳 중 6곳이 해안이라, 바다를 안 지우면 기준선이 차가워져 ΔT가 부풀려진다.

In [1]:
import os, re, glob, warnings
import pandas as pd, numpy as np, geopandas as gpd, rasterio
from rasterio.windows import from_bounds
from rasterio.features import geometry_mask
from pyproj import Transformer
from shapely.ops import transform as shp_transform
from shapely.geometry import Point as _P
warnings.filterwarnings('ignore')
pd.set_option('display.width', 230)

# 작업 폴더 — 자료가 있는 곳으로 바꾸세요
os.chdir('C:/cross_the_street/docs/research/_data-center/기획서_꾸러미/정보공개청구')
LSD = 'LANDSAT'

ST_SCALE, ST_OFF = 0.00341802, 149.0      # ST_B10 DN → Kelvin
STQA_SCALE = 0.01                          # ST_QA DN → Kelvin(불확실도)
QA_BITS = {'fill':0,'dilated':1,'cirrus':2,'cloud':3,'shadow':4,'snow':5,'clear':6,'water':7}

print('rasterio', rasterio.__version__, '| geopandas', gpd.__version__)
print('LANDSAT 폴더:', os.path.abspath(LSD), '| 존재:', os.path.isdir(LSD))

rasterio 1.4.4 | geopandas 1.1.2
LANDSAT 폴더: C:\cross_the_street\docs\research\_data-center\기획서_꾸러미\정보공개청구\LANDSAT | 존재: True


---
## 2. 장면 인벤토리

파일명에서 위성·Path/Row·촬영일·품질등급(Tier)을 읽고, 4파일 세트가 갖춰진 장면만 남긴다.

파일명 규칙: `LC08_L2SP_114035_20200820_20200905_02_T1_ST_B10.TIF`
→ `LC08`(위성) `L2SP`(Level-2 Science Product) `114`(Path) `035`(Row) `20200820`(촬영일) `20200905`(처리일) `02`(Collection) `T1`(Tier 1)

In [2]:
# ── §14-a. 장면 인벤토리 ──
import os, re, glob, numpy as np, rasterio
from rasterio.windows import from_bounds
from pyproj import Transformer
LSD='LANDSAT'                      # 원본 위성영상 폴더 (git 제외)

ST_SCALE, ST_OFF = 0.00341802, 149.0     # ST_B10 DN → Kelvin
STQA_SCALE = 0.01                         # ST_QA DN → Kelvin(불확실도)
QA_BITS = {'fill':0,'dilated':1,'cirrus':2,'cloud':3,'shadow':4,'snow':5,'clear':6,'water':7}

def scene_list():
    out=[]
    for p in sorted(glob.glob(f'{LSD}/*_ST_B10.TIF')):
        b=p[:-11]; nm=b.replace('\\','/').split('/')[-1]
        m=re.search(r'(LC0[89])_L2SP_(\d{3})(\d{3})_(\d{8})_\d{8}_02_(T\d)$', nm)
        if m and all(os.path.exists(b+s) for s in ['_QA_PIXEL.TIF','_ST_QA.TIF','_MTL.txt']):
            out.append(dict(base=b,sat=m.group(1),path=m.group(2),row=m.group(3),
                            date=pd.to_datetime(m.group(4)),tier=m.group(5)))
    return pd.DataFrame(out)

def mtl_meta(base):
    """MTL.txt → 촬영시각(UTC)·구름량. KST = UTC+9 (Landsat 통과 ≈ 오전 11시)"""
    d={}
    for ln in open(base+'_MTL.txt',encoding='utf-8',errors='ignore'):
        if '=' not in ln: continue
        k,v=[x.strip().strip('"') for x in ln.split('=',1)]
        if k in ('SCENE_CENTER_TIME','DATE_ACQUIRED','CLOUD_COVER','CLOUD_COVER_LAND'): d[k]=v
    return d

SC=scene_list()
print('장면:',len(SC),' 위성:',SC.sat.value_counts().to_dict(),' Tier:',SC.tier.value_counts().to_dict())
print('Path/Row별:',{f'{p}/{r}':n for (p,r),n in SC.groupby(['path','row']).size().items()})
print('월별:',SC.date.dt.month.value_counts().sort_index().to_dict(),'| 연도별:',SC.date.dt.year.value_counts().sort_index().to_dict())
if len(SC):
    _m=mtl_meta(SC.iloc[0].base); _h=int(_m['SCENE_CENTER_TIME'][:2])+9
    print(f"촬영시각 예: {_m['SCENE_CENTER_TIME'][:8]} UTC = {_h}시 KST → ⚠ 오전 통과(일최고기온 14~16시 아님)")

장면: 101  위성: {'LC08': 53, 'LC09': 48}  Tier: {'T1': 97, 'T2': 4}
Path/Row별: {'114/034': 20, '114/035': 17, '114/036': 15, '115/034': 12, '115/035': 10, '115/036': 15, '116/034': 12}
월별: {6: 34, 7: 16, 8: 40, 9: 11} | 연도별: {2020: 13, 2021: 4, 2022: 8, 2023: 20, 2024: 35, 2025: 21}
촬영시각 예: 01:58:44 UTC = 10시 KST → ⚠ 오전 통과(일최고기온 14~16시 아님)


In [5]:
SC.head()

,base,sat,path,row,date,tier
0,LANDSAT\LC08_L2SP_114034_20200820_20200905_02_T1,LC08,114,034,2020-08-20,T1
1,LANDSAT\LC08_L2SP_114034_20210620_20210629_02_T1,LC08,114,034,2021-06-20,T1
2,LANDSAT\LC08_L2SP_114034_20210807_20210811_02_T1,LC08,114,034,2021-08-07,T1
3,LANDSAT\LC08_L2SP_114034_20220810_20220818_02_T2,LC08,114,034,2022-08-10,T2
4,LANDSAT\LC08_L2SP_114034_20220826_20220927_02_T1,LC08,114,034,2022-08-26,T1


---
## 3. LST 변환 + 마스킹 + 링 추출

**설계 3원칙**

1. **동일 장면 안에서만 비교한다.** LST 절대값은 날짜·시각·대기에 따라 크게 변한다. 그래서 산단 주변 값에서 **같은 장면의 원거리 링(경계+10~20km) 중앙값**을 뺀 **ΔLST**만 쓴다. 같은 시각·같은 대기라 교란이 상쇄된다.
2. **바다·구름·그림자·눈·고불확실도를 지운다.** 위 QA 비트 + `ST_QA > 4K`.
3. **링은 폴리곤 경계 바깥 거리로 잰다.** 열은 산단 '중심'이 아니라 **산업시설 경계**에서 퍼진다. 울산미포처럼 띠 모양이면 중심 반경은 산단 안팎을 뒤섞는다. 실폴리곤이 없는 곳만 점 버퍼를 쓴다.

> ⚠ **성능** — 전 장면은 약 7,800×7,700 = 6천만 화소다. 대상 주변 window만 읽는다. 전체를 읽으면 사실상 끝나지 않는다.
> ⚠ **no-data 모서리** — Landsat 장면은 직사각 GeoTIFF 안의 *기울어진 평행사변형*이다. 좌표가 bbox 안이어도 촬영범위 밖일 수 있어, '장면 밖'과 '구름'을 구분해 사유를 돌려준다.

In [6]:
# ── §14-b. LST 변환 + 마스킹 + 링 추출 (형상 기반) ──
# [정정 07-20] 이전 판은 좌표를 손으로 근사 입력해 실제 산단에서 1.1~8.2km 빗나갔다(당진 8.2·포항 3.1·여수 3.1).
#   0~1km 링을 재는데 3km를 빗나가면 다른 장소를 측정한 것 → 여수가 -2.04℃(음수)로 나온 원인.
#   이제 §7과 같은 PDAN/YUCH 실형상에서 코드로 유도한다. 손입력 좌표 금지.
# [정정 07-20] 링 기준도 '중심점 반경'에서 '폴리곤 경계 바깥 거리'로 변경 — §10-b의 교훈(열은 중심이 아니라 산업시설
#   경계에서 퍼진다. 울산미포처럼 벨트형이면 중심 반경은 산단 안팎을 뒤섞는다). YUCH 폴리곤이 없는 곳만 점 버퍼.
from rasterio.features import geometry_mask
from shapely.ops import transform as shp_transform

def _to_scene(geom, src_epsg, dst_crs):
    tf=Transformer.from_crs(src_epsg, dst_crs, always_xy=True).transform
    return shp_transform(tf, geom)

RINGS=[(0,1000),(1000,2000),(2000,3000),(3000,5000),(5000,10000)]   # 경계 바깥 거리(m)
REF=(10000,20000)        # 동일장면 기준선 — 같은 시각·같은 대기라 교란 상쇄

def site_scene_dT(base, geom5179, is_poly, min_ref=5000, min_px=50, unc_max=4.0):
    # geom5179: YUCH union 폴리곤(또는 점) — EPSG:5179. is_poly=True면 R0(폴리곤 내부)도 산출.
    with rasterio.open(base+'_ST_B10.TIF') as s:
        g = _to_scene(geom5179, 5179, s.crs)
        gb = g.bounds
        if not (s.bounds.left < (gb[0]+gb[2])/2 < s.bounds.right and
                s.bounds.bottom < (gb[1]+gb[3])/2 < s.bounds.top): return {'사유':'장면 bbox 밖'}
        w  = from_bounds(gb[0]-REF[1]-1000, gb[1]-REF[1]-1000, gb[2]+REF[1]+1000, gb[3]+REF[1]+1000, s.transform)
        st = s.read(1,window=w,boundless=True,fill_value=0).astype('float64')
        tr = s.window_transform(w); shp=st.shape
    with rasterio.open(base+'_QA_PIXEL.TIF') as s: qa=s.read(1,window=w,boundless=True,fill_value=1)
    with rasterio.open(base+'_ST_QA.TIF')   as s: sq=s.read(1,window=w,boundless=True,fill_value=0).astype('float64')

    lst = st*ST_SCALE + ST_OFF - 273.15                       # DN → ℃
    bad = (st==0)                                             # fill(무자료)
    for k in ('fill','dilated','cirrus','cloud','shadow','snow'): bad |= ((qa>>QA_BITS[k])&1)>0
    bad |= ((qa>>QA_BITS['water'])&1)>0                       # ★ 바다 제거 — 해안 산단 6곳 필수
    bad |= (sq*STQA_SCALE > unc_max)                          # 불확실도 4K 초과 제거
    lst = np.where(bad, np.nan, lst)

    inside=lambda gg: geometry_mask([gg], out_shape=shp, transform=tr, invert=True)
    core = inside(g)
    # Landsat 장면은 직사각 GeoTIFF 안의 '기울어진 평행사변형' — bbox 안이라도 촬영범위 밖(no-data 모서리)일 수 있다.
    if core.any() and (st[core]==0).mean()>0.9: return {'사유':'촬영범위 밖(no-data 모서리)'}

    ringmask={}
    prev=core if is_poly else None
    for a,b in RINGS:
        outer=inside(g.buffer(b)); inner=inside(g.buffer(a)) if a>0 else (core if is_poly else np.zeros(shp,bool))
        ringmask[f'{a//1000}-{b//1000}km']=outer&~inner
    rf=inside(g.buffer(REF[1]))&~inside(g.buffer(REF[0]))

    ref=lst[rf]; ref=ref[~np.isnan(ref)]
    if len(ref)<min_ref: return {'사유':'기준선 유효화소 부족(구름·바다)'}
    rb=float(np.median(ref)); out={'사유':'OK','기준선C':round(rb,1),'기준n':len(ref)}
    if is_poly:
        v=lst[core]; v=v[~np.isnan(v)]
        out['ΔR0내부']=round(float(np.mean(v))-rb,2) if len(v)>=min_px else np.nan
    for lab,mk in ringmask.items():
        v=lst[mk]; v=v[~np.isnan(v)]
        out['Δ'+lab]=round(float(np.mean(v))-rb,2) if len(v)>=min_px else np.nan
        out['유효'+lab]=round(len(v)/max(mk.sum(),1),2)
    return out
print('함수 정의 완료: site_scene_dT (형상 기반 · YUCH 폴리곤 우선)')

함수 정의 완료: site_scene_dT (형상 기반 · YUCH 폴리곤 우선)


---
## 4. 대상 형상 — 좌표를 손으로 넣지 않는다

산단 위치는 **ILIS shapefile에서 코드로 유도**한다. 유치업종 폴리곤(YUCH)이 있으면 그것을, 없으면 대표점(PDAN)을 쓴다.

> ⚠ **왜 이렇게 하나** — 초판에서 좌표를 손으로 근사 입력했다가 실제 산단에서 **1.1~8.2km** 빗나갔다(당진 8.2 · 포항 3.1 · 여수 3.1km). 0~1km 링을 재는데 3km를 빗나가면 완전히 다른 장소를 측정한 것이다. 증상은 여수가 **−2.04℃**(음수)로 나온 것이었다.
> ⚠ **광양 주의** — PDAN 대표점이 광양만 **바다** 위에 찍혀 있다(위성영상으로 확인). YUCH 폴리곤(제철소 본체)을 쓰면 해결된다.

In [8]:
# 산업단지 형상 — ILIS shapefile (EPSG:5186)
PDAN = gpd.read_file('0718_DAM_PDAN/DAM_PDAN.shp', encoding='EUC-KR')
YUCH = gpd.read_file('0718_DAM_YUCH/DAM_YUCH.shp', encoding='EUC-KR')
_p4 = PDAN.to_crs(4326); PDAN['lon'] = _p4.geometry.x; PDAN['lat'] = _p4.geometry.y
print(f'PDAN 대표점 {len(PDAN):,}개 · YUCH 폴리곤 {len(YUCH):,}개 · 원 CRS EPSG:{PDAN.crs.to_epsg()}')
print('단지유형:', PDAN['DANJI_TYPE'].value_counts().to_dict(), '(1국가 2일반 3도시첨단 4농공)')

PDAN 대표점 1,451개 · YUCH 폴리곤 17,438개 · 원 CRS EPSG:5186
단지유형: {'2': 819, '4': 486, '1': 93, '3': 53} (1국가 2일반 3도시첨단 4농공)


In [9]:
# ── §14-c. 대상 형상 유도 + 부지별 ΔLST ──
from shapely.geometry import Point as _P
CLOUD_MAX=30.0
JJA_ONLY=True     # §3·§4와 동일하게 6~8월만. 9월 포함해도 0-1km ΔT 최대 0.14℃ 차 → 결론 불변

# (1) 8 산단 — §7 TG와 동일 정의. YUCH 실폴리곤 우선, 없으면 PDAN 점.
_TG=[('광양·제철','광양','1'),('여수·석화','여수','1'),('울산미포','울산·미포','1'),('온산','온산','1'),
     ('포항·제철','포항','1'),('구미·전자','구미(2·3단지)','1'),('당진1철강','당진1철강','2'),('동해북평','북평','1')]
GEO={}
for lbl,nm,ty in _TG:
    r=PDAN[(PDAN.DAN_NAME==nm)&(PDAN.DANJI_TYPE==ty)].iloc[0]
    yc=YUCH[YUCH.DAN_ID==r.DAN_ID]
    if len(yc): GEO[lbl]=(yc.to_crs(5179).geometry.union_all(), True)          # 실폴리곤
    else:       GEO[lbl]=(gpd.GeoSeries([_P(r.lon,r.lat)],crs=4326).to_crs(5179).iloc[0], False)  # 점
# (2) 개별 폐공장 부지 (§13, V-World 지오코딩) — 폴리곤 없음 → 점
for nm,lo,la in [('현대제철 인천',126.64432,37.48593),('동국제강 인천',126.64489,37.48324),('심팩 포항',129.37480,35.99267)]:
    GEO[nm]=(gpd.GeoSeries([_P(lo,la)],crs=4326).to_crs(5179).iloc[0], False)
print('대상:',len(GEO),'| 폴리곤 기준:',[k for k,(g,p) in GEO.items() if p])
print('점 기준(YUCH 없음·개별부지):',[k for k,(g,p) in GEO.items() if not p])

rec=[]; why=[]
for nm,(g,is_poly) in GEO.items():
    for _,s in SC.iterrows():
        if JJA_ONLY and s.date.month not in (6,7,8): continue
        m=mtl_meta(s.base)
        if float(m.get('CLOUD_COVER',100))>CLOUD_MAX:
            why.append((nm,f'{s.path}/{s.row}','장면 구름>%d%%'%CLOUD_MAX)); continue
        r=site_scene_dT(s.base,g,is_poly)
        why.append((nm,f'{s.path}/{s.row}',r['사유']))
        if r['사유']!='OK': continue
        rec.append(dict(부지=nm,형상='폴리곤' if is_poly else '점',날짜=s.date.date(),위성=s.sat,tier=s.tier,
                        구름pct=float(m.get('CLOUD_COVER',np.nan)),KST=f"{int(m['SCENE_CENTER_TIME'][:2])+9}시",
                        **{k:v for k,v in r.items() if k!='사유'}))
LST=pd.DataFrame(rec)
_dc=(['ΔR0내부'] if 'ΔR0내부' in LST.columns else [])+[f'Δ{a//1000}-{b//1000}km' for a,b in RINGS]

# 커버리지 진단 — 어느 부지가 왜 비었나 (다운로드 진행 중 판단용)
W=pd.DataFrame(why,columns=['부지','PathRow','사유'])
print('\n=== 커버리지 진단 (부지 × 사유) ===')
print(W.pivot_table(index='부지',columns='사유',aggfunc='size',fill_value=0).to_string())
_none=[n for n in GEO if n not in set(LST.부지)] if len(LST) else list(GEO)
if _none: print('\n⚠ 아직 유효장면 0인 부지:',_none)

if len(LST):
    agg=LST.groupby(['부지','형상']).agg(장면=('날짜','size'),**{c:(c,'mean') for c in _dc}).round(2)
    print(f'\n=== 부지별 평균 ΔLST (℃, 기준=동일장면 경계+10~20km 링 중앙값) ===')
    print(agg.sort_values('Δ0-1km',ascending=False).to_string())
    print('\n=== 장면별 원자료 (재현·이상치 확인용) ===')
    print(LST[['부지','형상','날짜','위성','KST','구름pct','기준선C']+_dc].to_string(index=False))
    LST.to_csv('DERIVED_0720_산단_LST_반경별.csv',index=False,encoding='utf-8-sig')
    print('\n저장: DERIVED_0720_산단_LST_반경별.csv')
    print('\n⚠ 해석 주의')
    print('  1) ★ 점 기준과 폴리곤 기준의 Δ0-1km는 의미가 다르다 — 나란히 비교 금지.')
    print('     · 폴리곤 부지: ΔR0내부=공장부지 자체, Δ0-1km=경계 바깥 0~1km (§10-b 인구 컬럼과 같은 규약)')
    print('     · 점 부지(YUCH 없음): Δ0-1km가 사실상 공장부지 안 → 폴리곤의 ΔR0내부와 대응시켜 읽을 것')
    print('  2) 울산미포는 도심이 산단에 붙어 원거리 링도 뜨겁다 → ΔT 과소평가 방향')
    print('  3) ΔT는 지표면온도 차이지 기온 차이가 아니다 — ×1.6/℃ 곡선에 직접 대입 금지(아래 해석 참조)')

대상: 11 | 폴리곤 기준: ['광양·제철', '울산미포', '온산', '구미·전자', '당진1철강', '동해북평']
점 기준(YUCH 없음·개별부지): ['여수·석화', '포항·제철', '현대제철 인천', '동국제강 인천', '심팩 포항']

=== 커버리지 진단 (부지 × 사유) ===
사유       OK  기준선 유효화소 부족(구름·바다)  장면 bbox 밖  촬영범위 밖(no-data 모서리)
부지                                                             
광양·제철    13                   0         63                   14
구미·전자    25                   0         65                    0
당진1철강    10                   1         70                    9
동국제강 인천  10                   1         79                    0
동해북평     26                   1         63                    0
심팩 포항    16                   0         74                    0
여수·석화    13                   0         63                   14
온산       26                   4         60                    0
울산미포     16                   0         60                   14
포항·제철    16                   0         74                    0
현대제철 인천  10                   1         79                    0

===

---
## 5. 결과 읽는 법

- `ΔR0내부` = 폴리곤 **내부**(공장부지 자체)
- `Δ0-1km` 이후 = 경계 **바깥** 거리

**★ 점 기준과 폴리곤 기준의 `Δ0-1km`는 의미가 다르다.** 폴리곤 부지는 경계 바깥이고, 점 부지(YUCH 없는 포항·여수·개별 공장)는 사실상 공장 안이다. 비교하려면 **점의 `Δ0-1km` ↔ 폴리곤의 `ΔR0내부`** 로 짝지어야 한다.

### 확인된 결과 (101장 · 2020~2025 여름 · 구름 30% 이하)

| 부지 | 기준 | 공장부지 자체 | 5~10km |
|---|---|---:|---:|
| 포항·제철 | 점 | **+17.3℃** | +3.1 |
| 심팩 포항 | 점 | +14.3 | +2.1 |
| 구미·전자 | 폴리곤 | +13.1 | +2.3 |
| 당진1철강 | 폴리곤 | +12.6 | +1.1 |
| 울산미포 | 폴리곤 | +12.1 | +1.9 |
| 온산 | 폴리곤 | +11.2 | +2.9 |
| 동해북평 | 폴리곤 | +10.9 | +2.0 |
| 광양·제철 | 폴리곤 | +10.2 | +0.8 |
| 여수·석화 | 점 | +8.5 | +2.2 |
| 현대제철 인천 | 점 | +5.9 | +0.6 |
| 동국제강 인천 | 점 | +5.5 | +0.6 |

거리에 따른 **단조 감쇠**가 모든 부지에서 재현된다. 장면간 표준편차는 대체로 1~3℃.

**남은 편향**: 기준선(경계+10~20km)이 대개 산림·농경지라 이 값에는 *산단 vs 자연*이 섞여 있다. *산단 vs 주거지* 대비였다면 더 작았을 것이므로 **상한 성격**이다.

---
## 6. 지표면온도 → 기온 환산 (선택)

§5의 ΔLST는 **지표면** 온도차다. 사람이 느끼는 **기온** 차로 옮기려면 환산계수가 필요하다. 외부 상수를 빌리는 대신 우리 자료로 추정한다.

**설계**: 같은 장면 안 여러 관측소에서 `기온 ~ 장면고정효과 + β·LST`. 장면 고정효과가 '그날 그 지역의 더위'를 흡수하므로 β는 **순수한 공간 기울기**가 된다. 표준오차는 관측소 단위로 묶는다.

> 이 절은 관측소 자료가 있어야 돌아간다. 없으면 건너뛰어도 §5까지의 결과는 유효하다.

In [10]:
# ── §15-a. 관측망 로더 (ASOS·AWS 공통) ──
import glob, os
# 두 관측망을 같은 함수로 처리한다. 파일 형식이 동일(지점·시작일·종료일·지점명·위도·경도 + 일자료 3기온).
NETS={'ASOS':dict(stn='OPEN_0720_기상청_ASOS지점정보.csv',
                  day=sorted(glob.glob('OPEN_0522_기상청_ASOS일자료_*_summer.csv'))),
      'AWS' :dict(stn='OPEN_0720_기상청_AWS지점정보.csv',
                  # [갱신 07-21] 459지점 JJA 파일(연도별 6개)로 교체 — 기존 261지점보다 넓다. 없으면 옛 파일 fallback.
                    day=sorted(glob.glob('ASOS+AWS/OBS_AWS_DD_*.csv')) or ['OPEN_0720_기상청_AWS일자료_summer.csv'])}

def _rd(p):
    for e in ('cp949','utf-8-sig','utf-8'):
        try:
            d=pd.read_csv(p,encoding=e); d.columns=[str(c).strip() for c in d.columns]; return d
        except Exception: continue
    raise IOError(p)

def load_net(name):
    cfg=NETS[name]
    if not os.path.exists(cfg['stn']) or not all(os.path.exists(f) for f in cfg['day']):
        print(f'⏸ {name}: 파일 없음 — 건너뜀'); return None,None
    s=_rd(cfg['stn'])
    s=s.rename(columns={c:'lat' for c in s.columns if '위도' in c}|{c:'lon' for c in s.columns if '경도' in c})
    for c in ['지점','lat','lon']: s[c]=pd.to_numeric(s[c],errors='coerce')
    # [중요] 지점정보는 이전(relocation)·번호 재할당 이력이 여러 행으로 들어있다.
    #   ASOS 천안 15.4km · AWS 최대 417km(= 이동이 아니라 지점번호 재사용).
    #   최신 행을 그냥 쓰면 과거 날짜의 위성 화소를 엉뚱한 곳에서 뽑는다(§14 좌표 이탈과 같은 실수).
    s['시작일']=pd.to_datetime(s.get('시작일'),errors='coerce').fillna(pd.Timestamp('1900-01-01'))
    s['종료일']=pd.to_datetime(s.get('종료일'),errors='coerce')
    s=s.dropna(subset=['지점','lat','lon'])[['지점','lat','lon','시작일','종료일']].reset_index(drop=True)

    d=pd.concat([_rd(f) for f in cfg['day']],ignore_index=True)
    d=d.rename(columns={'평균기온(°C)':'Tavg','최고기온(°C)':'Tmax','최저기온(°C)':'Tmin'})
    d['date']=pd.to_datetime(d['일시'],errors='coerce').dt.date
    d['지점']=pd.to_numeric(d['지점'],errors='coerce')
    d=d[['지점','지점명','date','Tavg','Tmax','Tmin']].dropna(subset=['date','지점'])
    d=d[pd.to_datetime(d.date).dt.month.isin([6,7,8])]          # JJA만 (§3·§4와 동일 규율)
    nmulti=(s.groupby('지점').size()>1).sum()
    print(f'{name}: 지점 {s.지점.nunique()}개(정보 {len(s)}행, 이전이력 {nmulti}개) · 일자료 {len(d):,}행(JJA) · 관측지점 {d.지점.nunique()}개')
    return s,d

def stn_on(stn, day):
    # day에 유효한 좌표만 — 시작일 ≤ day ≤ 종료일(없으면 현재까지)
    t=pd.Timestamp(day)
    m=stn[(stn.시작일<=t)&(stn.종료일.isna()|(stn.종료일>=t))]
    return m.sort_values('시작일').drop_duplicates('지점',keep='last')[['지점','lat','lon']].reset_index(drop=True)

NET={}
for _n in NETS:
    _s,_d=load_net(_n)
    if _s is not None: NET[_n]=(_s,_d)
print('\n사용 가능 관측망:',list(NET))

ASOS: 지점 105개(정보 148행, 이전이력 37개) · 일자료 52,985행(JJA) · 관측지점 97개
AWS: 지점 576개(정보 2518행, 이전이력 538개) · 일자료 134,535행(JJA) · 관측지점 261개

사용 가능 관측망: ['ASOS', 'AWS']


In [11]:
# ── §15-b. 관측소 화소 LST 추출 (지점당 소형 window) ──
from shapely.geometry import Point as _P
RADII=[30,100,500,1000]        # 관측소 주변 평균 반경(m) — 유효 footprint 탐색
# [성능 정정 07-20] 초판은 '전 관측소를 감싸는 하나의 거대 window'(장면 전역 ~7000×7000)를 읽고
#   지점마다 그 위에서 거리 배열을 다시 계산했다 → 사실상 끝나지 않음. 지점당 작은 window만 읽도록 교체.
def lst_at_points(base, lats, lons, radii=RADII, unc_max=4.0, min_frac=0.3):
    n=len(lats); out={R:(np.full(n,np.nan),np.zeros(n)) for R in radii}
    rmax=max(radii)+60
    with rasterio.open(base+'_ST_B10.TIF') as s0:
        xs,ys=Transformer.from_crs(4326,s0.crs,always_xy=True).transform(list(lons),list(lats))
        xs,ys=np.asarray(xs),np.asarray(ys); b=s0.bounds
        ok=np.where((xs>b.left+rmax)&(xs<b.right-rmax)&(ys>b.bottom+rmax)&(ys<b.top-rmax))[0]
        if len(ok)==0: return out,0
        tr0=s0.transform
        with rasterio.open(base+'_QA_PIXEL.TIF') as s1, rasterio.open(base+'_ST_QA.TIF') as s2:
            for j in ok:
                w=from_bounds(xs[j]-rmax,ys[j]-rmax,xs[j]+rmax,ys[j]+rmax,tr0)
                st=s0.read(1,window=w,boundless=True,fill_value=0).astype('float64')
                if st.size==0 or (st==0).all(): continue
                qa=s1.read(1,window=w,boundless=True,fill_value=1)
                sq=s2.read(1,window=w,boundless=True,fill_value=0).astype('float64')
                lst=st*ST_SCALE+ST_OFF-273.15
                bad=(st==0)
                for k in ('fill','dilated','cirrus','cloud','shadow','snow'): bad|=((qa>>QA_BITS[k])&1)>0
                bad|=((qa>>QA_BITS['water'])&1)>0
                bad|=(sq*STQA_SCALE>unc_max)
                lst=np.where(bad,np.nan,lst)
                tr=s0.window_transform(w); ny,nx=lst.shape
                cc,rr=np.meshgrid(np.arange(nx)+0.5,np.arange(ny)+0.5)
                px,py=tr*(cc,rr); d2=(px-xs[j])**2+(py-ys[j])**2
                for R in radii:
                    m=d2<=R*R
                    if not m.any(): continue
                    g=lst[m]; g=g[~np.isnan(g)]
                    fr=len(g)/m.sum(); out[R][1][j]=fr
                    if fr>=min_frac and len(g): out[R][0][j]=g.mean()
    return out,len(ok)
print('함수 정의 완료: lst_at_points (지점당 소형 window)')

함수 정의 완료: lst_at_points (지점당 소형 window)


In [12]:
# ── §15-c. (관측소 × 장면) 패널 — 관측망별 ──
def build_panel(name, stn, day):
    rows=[]
    for _,s in SC.iterrows():
        if JJA_ONLY and s.date.month not in (6,7,8): continue
        if float(mtl_meta(s.base).get('CLOUD_COVER',100))>CLOUD_MAX: continue
        cs=stn_on(stn,s.date.date())
        if not len(cs): continue
        r,nin=lst_at_points(s.base,cs.lat.values,cs.lon.values)
        if nin==0: continue
        base=dict(scene=os.path.basename(s.base),date=s.date.date(),path=s.path,row=s.row)
        for j in range(len(cs)):
            if all(np.isnan(r[R][0][j]) for R in RADII): continue
            rows.append({**base,'지점':int(cs.지점[j]),**{f'LST_{R}m':r[R][0][j] for R in RADII}})
    P=pd.DataFrame(rows)
    if not len(P): print(f'{name}: 유효 쌍 0'); return P
    P=P.merge(day[['지점','지점명','date','Tavg','Tmax','Tmin']],on=['지점','date'],how='inner')
    P['net']=name; P['scene_id']=P['scene']
    per=P.groupby('scene').size()
    print(f"\n[{name}] (관측소×장면) 쌍 {len(P):,} · 관측소 {P.지점.nunique()}개 · 장면 {P.scene.nunique()}개")
    print(f"  장면당 관측소: 중앙값 {per.median():.0f} (최소 {per.min()} · 최대 {per.max()})")
    print(f"  ★ 장면 내 관측소간 LST 표준편차(회귀 leverage) 중앙값:")
    for R in RADII:
        print(f"     반경 {R:>4}m: {P.groupby('scene')[f'LST_{R}m'].std().median():5.2f}℃  (유효 {P[f'LST_{R}m'].notna().mean()*100:3.0f}%)")
    print(f"  기온 SD 중앙값: Tmax {P.groupby('scene')['Tmax'].std().median():.2f}℃ · Tavg {P.groupby('scene')['Tavg'].std().median():.2f}℃")
    return P

PANEL={n:build_panel(n,*NET[n]) for n in NET}
SP=pd.concat([p for p in PANEL.values() if len(p)],ignore_index=True) if PANEL else pd.DataFrame()


[ASOS] (관측소×장면) 쌍 905 · 관측소 83개 · 장면 85개
  장면당 관측소: 중앙값 11 (최소 1 · 최대 29)
  ★ 장면 내 관측소간 LST 표준편차(회귀 leverage) 중앙값:
     반경   30m:  3.00℃  (유효  90%)
     반경  100m:  2.88℃  (유효  92%)
     반경  500m:  2.68℃  (유효  93%)
     반경 1000m:  2.82℃  (유효  97%)
  기온 SD 중앙값: Tmax 1.38℃ · Tavg 0.96℃

[AWS] (관측소×장면) 쌍 1,792 · 관측소 206개 · 장면 76개
  장면당 관측소: 중앙값 20 (최소 1 · 최대 63)
  ★ 장면 내 관측소간 LST 표준편차(회귀 leverage) 중앙값:
     반경   30m:  3.66℃  (유효  86%)
     반경  100m:  3.56℃  (유효  86%)
     반경  500m:  3.56℃  (유효  92%)
     반경 1000m:  3.38℃  (유효  98%)
  기온 SD 중앙값: Tmax 1.77℃ · Tavg 1.28℃


In [18]:
# ── §15-d. β 추정 — 관측망별 · 반경별 · 기온지표별 ──
import statsmodels.formula.api as smf
BETA={}
if not len(SP):
    print('⏸ §15-c 선행 필요')
else:
    print(f"{'관측망':7}{'기온':6}{'반경':>7}{'β(기온℃/지표℃)':>16}{'95%CI':>18}{'p':>9}{'n':>7}{'지점':>6}")
    for net in PANEL:
        P=PANEL[net]
        if not len(P): continue
        for tv in ['Tmax','Tavg']:
            for R in RADII:
                d=P[['scene_id','지점',tv,f'LST_{R}m']].dropna().rename(columns={f'LST_{R}m':'LST'})
                if len(d)<100 or d.scene_id.nunique()<5: continue
                m=smf.ols(f'{tv} ~ LST + C(scene_id)',data=d).fit(cov_type='cluster',cov_kwds={'groups':d['지점']})
                b=m.params['LST']; ci=m.conf_int().loc['LST']; p=m.pvalues['LST']
                BETA[(net,tv,R)]=(b,ci[0],ci[1],len(d),d.지점.nunique())
                print(f"{net:7}{tv:6}{R:>6}m{b:>16.3f}{f'{ci[0]:.3f}~{ci[1]:.3f}':>18}{p:>9.1e}{len(d):>7}{d.지점.nunique():>6}")
    if BETA:
        # 채택: 관측소 수가 가장 많은 관측망 · Tmax · R²가 아니라 CI 폭이 가장 좁은 반경(=식별력 최대)
        cands=[k for k in BETA if k[1]=='Tmax']
        best=min(cands,key=lambda k:(BETA[k][2]-BETA[k][1])/max(abs(BETA[k][0]),1e-9))
        b,lo,hi,n,ns=BETA[best]
        print(f'\n★ 채택 β = {b:.3f} ℃기온/℃지표  (95%CI {lo:.3f}~{hi:.3f})')
        print(f'   관측망 {best[0]} · {best[1]} · 반경 {best[2]}m · n={n:,} · 지점 {ns}개')
        if len([k for k in BETA if k[1]=='Tmax'])>1:
            print('\n관측망간 일치도(Tmax, 같은 반경) — 크게 다르면 어느 한쪽 편의 의심:')
            for R in RADII:
                vs={net:BETA[(net,'Tmax',R)][0] for net in PANEL if (net,'Tmax',R) in BETA}
                if len(vs)>1: print(f"   반경 {R:>4}m: "+" · ".join(f"{k} {v:+.3f}" for k,v in vs.items()))
        print(f'\n=== 산단 지표 ΔT → 기온 ΔT 환산 ===')
        # [정정 07-21] 폴리곤=R0만·점=0-1km (fillna 오염 제거)
        _c=LST.copy()
        _facmap={k:(g['ΔR0내부'].mean() if g['ΔR0내부'].notna().any() else g['Δ0-1km'].mean()) for k,g in LST.groupby('부지')}
        _c['공장부지ΔLST']=_c['부지'].map(_facmap)
        print(f"{'부지':14}{'지표ΔT':>8}{'기온ΔT(추정)':>13}{'95%CI':>17}")
        for k,v in _c.groupby('부지')['공장부지ΔLST'].mean().sort_values(ascending=False).items():
            print(f"{k:14}{v:>7.1f}℃{v*b:>12.2f}℃{f'{v*lo:.2f}~{v*hi:.2f}':>17}")


관측망    기온         반경      β(기온℃/지표℃)             95%CI        p      n    지점
ASOS   Tmax      30m           0.181       0.112~0.249  2.5e-07    816    82
ASOS   Tmax     100m           0.181       0.114~0.248  1.2e-07    831    83
ASOS   Tmax     500m           0.140       0.066~0.214  2.2e-04    839    82
ASOS   Tmax    1000m           0.138       0.058~0.218  7.3e-04    874    82
ASOS   Tavg      30m           0.107       0.056~0.158  4.5e-05    815    82
ASOS   Tavg     100m           0.111       0.059~0.163  3.3e-05    830    83
ASOS   Tavg     500m           0.159       0.100~0.218  1.2e-07    838    82
ASOS   Tavg    1000m           0.197       0.149~0.244  4.4e-16    873    82
AWS    Tmax      30m           0.281       0.237~0.325  5.5e-36   1534   191
AWS    Tmax     100m           0.293       0.249~0.337  4.4e-39   1543   191
AWS    Tmax     500m           0.293       0.240~0.346  1.4e-27   1640   200
AWS    Tmax    1000m           0.294       0.236~0.351  1.2e-23   1749   203

> ### ⚠ 위 「환산」 표는 단일 β 결과 — §15-e·§15-f가 정정한다  `[🔖 2026-07-21]`
>
> 바로 위 「산단 지표 ΔT → 기온 ΔT 환산」 표는 **AWS Tmax·반경 100m β=0.258 하나로만** 곱한 값이다 (포항 17.3 × 0.258 = 4.46℃). 이 표는 **한 점 추정**이고, 최종 해석은 아래 두 절이 정정한다:
>
> - **§15-e** — 두 관측망 β가 1.6배 어긋나는 원인 판별 → **회귀희석 기각 못 함.** 위 단일 β는 눌린 하한일 수 있어, 범위(0.181~0.258)와 희석보정값(0.459)을 함께 본다.
> - **§15-f** — 시각 불일치는 오차가 아니라 결합강도의 일변화. 일최고 β ≈ 14~15시 β라 **Tmax가 옳은 지표.**
> - **공통 한계** — 관측소는 개방·통풍 부지에만 있어 +17℃ 산단 적용은 외삽(상한 과대 방향).
>
> → 따라서 논증 문서의 산단별 기온 ΔT는 위 단일값이 아니라 **이 범위**로 제시한다 (예: 포항 +3.1~7.9℃).

### 6-b. 두 관측망 β가 어긋난다 — 원인 판별

ASOS β=0.181, AWS β=0.293으로 **1.6배** 차이가 난다. 하나를 고르기 전에 원인을 짚는다.

| 가설 | 검정 | 결과 |
|---|---|---|
| **회귀희석** — 관측 LST에 오차가 있으면 기울기가 0쪽으로 눌리고, LST 분산이 작은 ASOS가 더 눌린다 | 필요 측정오차를 역산해 실제 오차 규모와 대조 | **기각 못 함** |
| **AWS 설치 편의** — 옥상·도심 설치라 기온을 더 뜨겁게 읽는다 | 5km 이내 ASOS↔AWS 동일일자 일최고기온 대조 | **기각** (평균차 −0.05℃) |

회귀희석을 기각하지 못하므로 **두 직접 추정치 모두 참값보다 눌린 하한**으로 봐야 한다. 필요 측정오차(약 2.3℃)는 **반경만 30m↔500m로 바꿔도 LST가 그만큼 흔들리는** 규모라 실재한다.

In [17]:
# ── §15-e. 관측망 대조 + β 범위로 환산 (강건성) ──
from scipy.spatial import cKDTree
# (1) 회귀희석 가설 — 올바른 모수화로 필요 측정오차 역산
# [정정 07-20 · 2차] 이 블록은 두 번 틀렸다. 경위를 남긴다.
#   1차: 분모를 (r·vA-vW)로 써서 음수가 나왔는데 abs()로 가림. 값(2.31℃)은 우연히 맞았으나
#        "비현실적이므로 기각"이라는 결론이 하드코딩돼 있었다.
#   2차: 음수의 원인을 캐지 않고 모형을 바꿔 재유도 → λ=σ²_true/(σ²_true+e) 식에 '관측' SD를
#        집어넣었다. 그러면 β비에 vA/vW라는 하한이 생기는 것처럼 보이고, 그 하한을 근거로
#        또 "기각"했다. 존재하지 않는 하한을 만들어 결론을 맞춘 셈.
#   교훈: 결론(기각)을 먼저 정해두고 근거를 갈아끼웠다. 아래는 시뮬레이션으로 검증한 판.
#
# 올바른 식 — 우리가 잰 SD는 '관측' LST의 SD이므로 V = σ²_true + σ²_e 이고
#   β̂ = Cov(T, L_obs)/Var(L_obs) = β·(V−e)/V     (λ = (V−e)/V)
#   β_A/β_W = [(VA−e)/VA] / [(VW−e)/VW] 는 e가 커지면 0까지 내려간다 → 하한 없음.
#   e = VA·VW(1−r) / (VW − r·VA)
if ('ASOS','Tmax',100) in BETA and ('AWS','Tmax',100) in BETA:
    bA=BETA[('ASOS','Tmax',100)][0]; bW=BETA[('AWS','Tmax',100)][0]
    VA=PANEL['ASOS'].groupby('scene')['LST_100m'].std().median()**2
    VW=PANEL['AWS'].groupby('scene')['LST_100m'].std().median()**2
    r=bA/bW; e=VA*VW*(1-r)/(VW-r*VA)
    print(f'[가설①] 회귀희석 — 관측 LST 분산: ASOS {VA:.2f} / AWS {VW:.2f} (SD {VA**.5:.2f}/{VW**.5:.2f}℃)')
    print(f'  관측 β비 = {bA:.3f}/{bW:.3f} = {r:.4f}')
    if not (0 < e < VA):
        print(f'  → 유효한 측정오차 해 없음(e={e:.2f}) → 희석으로는 설명 불가, 가설 기각')
        BETA_DIS=None
    else:
        se=e**0.5
        print(f'  → 이를 정확히 설명하는 측정오차 σ_e = {se:.2f}℃ (검산 β비 {((VA-e)/VA)/((VW-e)/VW):.4f})')
        # 독립 대조: 반경 선택만 바꿔도 LST가 얼마나 흔들리나 = 공간 대표성 오차
        _rep=(SP['LST_30m']-SP['LST_500m']).dropna().std()
        print(f'  독립 대조 — 반경 30m vs 500m LST 차이 SD = {_rep:.2f}℃ (기온계가 느끼는 지표 규모를 모르는 데서 오는 대표성 오차)')
        print(f'  → 역산 {se:.2f}℃와 같은 자릿수 ⇒ **가설 기각 못 함**. 두 β 모두 눌린 값으로 봐야 한다.')
        BETA_DIS=(bA*VA/(VA-e), bW*VW/(VW-e))
        print(f'  희석 보정 β = {BETA_DIS[0]:.3f}(ASOS) / {BETA_DIS[1]:.3f}(AWS)')
        print(f'     ⚠ 두 값의 일치는 e를 그렇게 풀어서 생긴 것 — 독립 증거 아니다. σ_e 가정에 전적으로 의존.')

# (2) AWS 설치 편의 가설 — 5km 이내 동일일자 일최고기온 대조
_A=stn_on(NET['ASOS'][0],pd.Timestamp('2022-07-01')); _W=stn_on(NET['AWS'][0],pd.Timestamp('2022-07-01'))
_gA=gpd.GeoSeries(gpd.points_from_xy(_A.lon,_A.lat),crs=4326).to_crs(5179)
_gW=gpd.GeoSeries(gpd.points_from_xy(_W.lon,_W.lat),crs=4326).to_crs(5179)
_d,_i=cKDTree(np.c_[_gW.x,_gW.y]).query(np.c_[_gA.x,_gA.y])
_pair=pd.DataFrame({'asos':_A.지점,'aws':_W.지점.values[_i],'거리m':_d.round(0)})
_pair=_pair[_pair.거리m<=5000]
_m=(NET['ASOS'][1][['지점','date','Tmax']].rename(columns={'지점':'asos','Tmax':'T_asos'}).merge(_pair,on='asos')
      .merge(NET['AWS'][1][['지점','date','Tmax']].rename(columns={'지점':'aws','Tmax':'T_aws'}),on=['aws','date'])
      .dropna(subset=['T_asos','T_aws']))
_m['diff']=_m.T_aws-_m.T_asos
print(f"\n[가설②] AWS 설치 편의 — 5km 이내 짝 {_pair.shape[0]}개 중 동일일자 자료 있는 {_m.asos.nunique()}짝 · {len(_m):,}일")
print(f"  AWS − ASOS 일최고기온: 평균 {_m['diff'].mean():+.2f}℃ · 중앙 {_m['diff'].median():+.2f}℃ · SD {_m['diff'].std():.2f}")
print(f"  → 0 근처 = AWS가 체계적으로 뜨겁게 읽지 않음. 가설 기각")
print(f"  ⚠ 짝이 {_m.asos.nunique()}개뿐 — 표본 작음. AWS 지점을 더 받으면 재검정 권장")

# (3) β 범위로 환산 — 결론이 범위 전체에서 유지되나
# [정정 07-20] 헤드라인 범위는 반경 100m 고정(두 관측망이 같은 정의로 비교되는 지점).
#   반경까지 섞은 전 조합 envelope는 참고로 병기 — 반경 차이와 관측망 차이를 뭉뚱그리지 않는다.
_bl=BETA[('ASOS','Tmax',100)][0]; _bh=BETA[('AWS','Tmax',100)][0]
_el=min(BETA[k][0] for k in BETA if k[1]=='Tmax'); _eh=max(BETA[k][0] for k in BETA if k[1]=='Tmax')
print()
print("=== 산단 지표ΔT → 기온ΔT ===")
print(f"  헤드라인 β 범위(반경 100m): {_bl:.3f}(ASOS) ~ {_bh:.3f}(AWS)")
print(f"  참고 전 조합 envelope     : {_el:.3f} ~ {_eh:.3f}")
if BETA_DIS: print(f"  희석 보정 시(σ_e 가정 의존)  : {min(BETA_DIS):.3f} ~ {max(BETA_DIS):.3f}")
print()
print(f"{'부지':14}{'지표ΔT':>8}{'기온ΔT 하한':>12}{'기온ΔT 상한':>12}{'희석보정':>10}")
for k,v in _c.groupby('부지')['공장부지ΔLST'].mean().sort_values(ascending=False).items():
    _d=f"{v*max(BETA_DIS):>9.2f}℃" if BETA_DIS else f"{'-':>10}"
    print(f"{k:14}{v:>7.1f}℃{v*_bl:>11.2f}℃{v*_bh:>11.2f}℃"+_d)
print("\n\u2192 \uc9c0\uc11c\uba74\uc628\ub3c4\u2192\uae30\uc628 \ud574\uc11d\uc740 \ubcf8 \ub178\ud2b8\ubd81 \u00a77 \ud574\uc11d \uaddc\uc57d \ucc38\uc870")


[가설①] 회귀희석 — 관측 LST 분산: ASOS 8.27 / AWS 12.67 (SD 2.88/3.56℃)
  관측 β비 = 0.181/0.293 = 0.6173
  → 이를 정확히 설명하는 측정오차 σ_e = 2.30℃ (검산 β비 0.6173)
  독립 대조 — 반경 30m vs 500m LST 차이 SD = 2.35℃ (기온계가 느끼는 지표 규모를 모르는 데서 오는 대표성 오차)
  → 역산 2.30℃와 같은 자릿수 ⇒ **가설 기각 못 함**. 두 β 모두 눌린 값으로 봐야 한다.
  희석 보정 β = 0.503(ASOS) / 0.503(AWS)
     ⚠ 두 값의 일치는 e를 그렇게 풀어서 생긴 것 — 독립 증거 아니다. σ_e 가정에 전적으로 의존.

[가설②] AWS 설치 편의 — 5km 이내 짝 13개 중 동일일자 자료 있는 4짝 · 2,194일
  AWS − ASOS 일최고기온: 평균 -0.05℃ · 중앙 +0.20℃ · SD 1.06
  → 0 근처 = AWS가 체계적으로 뜨겁게 읽지 않음. 가설 기각
  ⚠ 짝이 4개뿐 — 표본 작음. AWS 지점을 더 받으면 재검정 권장

=== 산단 지표ΔT → 기온ΔT ===
  헤드라인 β 범위(반경 100m): 0.181(ASOS) ~ 0.293(AWS)
  참고 전 조합 envelope     : 0.138 ~ 0.294
  희석 보정 시(σ_e 가정 의존)  : 0.503 ~ 0.503

부지                지표ΔT     기온ΔT 하한     기온ΔT 상한      희석보정
포항·제철            17.3℃       3.12℃       5.06℃     8.69℃
심팩 포항            14.3℃       2.59℃       4.20℃     7.22℃
구미·전자            13.1℃       2.37℃       3.84℃     6.60℃
당진1철강            12.6℃       2.27℃       3.68℃     6.33℃


### 6-c. 시각별 β — 통과시각 기온을 쓰면 달라지나 *(시간자료 있을 때만)*

Landsat 통과는 10~11시 KST인데 일최고기온은 14~16시다. 이 차이가 **오차**라면 통과시각 기온을 쓸 때 두 관측망 β가 수렴해야 한다. 검정해 보면 그렇지 않다 — **β 자체가 오후로 갈수록 커진다**(결합강도의 일변화). 그리고 **일최고 β ≈ 14~15시 β**다.

→ 온열질환이 14~16시에 몰리므로 **일최고기온이 목적에 맞는 지표**다. 시각 불일치는 한계가 아니다.

*필요 자료*: `ASOS+AWS/OBS_{ASOS,AWS}_TIM_*.csv` (기상자료개방포털 → 시간자료). 없으면 이 셀은 건너뛴다.

In [15]:
# ── §15-f. 시각별 β (ASOS·AWS 시간자료) ──
# 자료: ASOS+AWS/OBS_{ASOS,AWS}_TIM_*.csv — 연도는 파일 내용에서 읽는다(파일명 순서 가정 금지).
def load_hourly(pat):
    fr=[]
    for f in sorted(glob.glob(pat)):
        for e in ('cp949','utf-8-sig','utf-8'):
            try: d=pd.read_csv(f,encoding=e); break
            except Exception: continue
        d.columns=[str(c).strip() for c in d.columns]
        d=d.rename(columns={[c for c in d.columns if '기온' in c][0]:'T'})
        d['dt']=pd.to_datetime(d['일시'],errors='coerce')
        fr.append(d[['지점','dt','T']].dropna(subset=['dt']))
    if not fr: return None
    h=pd.concat(fr,ignore_index=True); h['date']=h.dt.dt.date; h['hour']=h.dt.dt.hour
    return h

HR={n:load_hourly(f'ASOS+AWS/OBS_{n}_TIM_*.csv') for n in ['ASOS','AWS']}
HR={k:v for k,v in HR.items() if v is not None}
if not HR:
    print('⏸ 시간자료 없음 — ASOS+AWS/ 폴더 확인')
else:
    for k,v in HR.items(): print(f'{k} 시간자료: {len(v):,}행 · 지점 {v.지점.nunique()}개 · {v.date.min()}~{v.date.max()}')
    # 전 101장 통과시각이 KST 10.95~11.18시 → 11시 정시가 통과시각. 10·12·14·15시는 일변화 비교용.
    HOURS=[10,11,12,14,15]
    print(f"\n{'관측망':7}{'기온지표':>9}{'β':>9}{'95%CI':>18}{'SD':>8}{'n':>7}{'지점':>6}")
    BETA_HR={}
    for net in HR:
        P=SP[SP.net==net].copy()
        for h in HOURS:
            P=P.merge(HR[net][HR[net].hour==h][['지점','date','T']].rename(columns={'T':f'T{h}'}),
                      on=['지점','date'],how='left')
        Q=P.dropna(subset=['T11','Tmax'])         # ★ 동일 표본 — 시각 효과와 표본 효과 분리
        for y in [f'T{h}' for h in HOURS]+['Tmax']:
            d=Q[['scene','지점',y,'LST_100m']].dropna().rename(columns={y:'yv','LST_100m':'LST'})
            if len(d)<80: continue
            m=smf.ols('yv ~ LST + C(scene)',data=d).fit(cov_type='cluster',cov_kwds={'groups':d['지점']})
            b=m.params['LST']; ci=m.conf_int().loc['LST']; BETA_HR[(net,y)]=b
            print(f"{net:7}{y:>9}{b:>9.3f}{f'{ci[0]:.3f}~{ci[1]:.3f}':>18}"
                  f"{Q.groupby('scene')[y].std().median():>7.2f}℃{len(d):>7}{d.지점.nunique():>6}")
    print('\n관측망 β비 (1에 가까울수록 일치):')
    for y in [f'T{h}' for h in HOURS]+['Tmax']:
        if ('ASOS',y) in BETA_HR and ('AWS',y) in BETA_HR:
            print(f"  {y:>6}: {BETA_HR[('ASOS',y)]/BETA_HR[('AWS',y)]:.3f}")
print("\n\u2192 \ud574\uc11d\uc740 \u00a77 \ud574\uc11d \uaddc\uc57d \ucc38\uc870")


ASOS 시간자료: 1,270,026행 · 지점 97개 · 2020-06-01~2025-08-31
AWS 시간자료: 1,137,181행 · 지점 88개 · 2020-06-01~2025-08-31

관측망         기온지표        β             95%CI      SD      n    지점
ASOS         T10    0.059       0.016~0.101   1.07℃    831    83
ASOS         T11    0.114       0.072~0.156   1.10℃    831    83
ASOS         T12    0.129       0.078~0.181   1.13℃    831    83
ASOS         T14    0.182       0.109~0.255   1.47℃    831    83
ASOS         T15    0.180       0.097~0.264   1.62℃    831    83
ASOS        Tmax    0.181       0.114~0.248   1.38℃    831    83
AWS          T10    0.205       0.150~0.259   1.23℃    767    74
AWS          T11    0.226       0.177~0.275   1.33℃    768    74
AWS          T12    0.238       0.190~0.285   1.39℃    768    74
AWS          T14    0.292       0.232~0.352   1.65℃    768    74
AWS          T15    0.298       0.232~0.364   1.69℃    768    74
AWS         Tmax    0.275       0.227~0.322   1.51℃    768    74

관측망 β비 (1에 가까울수록 일치):
     T10: 0.287
     T

---
## 7. 해석 규약 — 여기까지만 말한다

### 측정된 것 `[사실]`
- 공장부지 지표가 주변 대비 **+5.5 ~ +17.3℃**, 거리 단조 감쇠. 11개 부지 전부 재현.
- 지표 1℃당 기온 **β = 0.181(ASOS) ~ 0.293(AWS)**. 측정오차를 보정하면 0.5까지 갈 수 있다(가정 의존).
- 환산: 가동 제철소 **+3~9℃**, 도심 폐공장 **+1~3℃**.

### 말하지 않는 것 `[해석 금지]`
1. **"데이터센터가 들어오면 +2℃ 오른다"** — 이 파이프라인은 *기존* 산업시설만 측정한다. 데이터센터 폐열의 승온은 별도 물리 계산이고, 제철소 열은 폐열 외에 공정열·복사가 섞여 기전이 다르다. 말할 수 있는 것은 **"+2℃ 가정이 실측된 산업 열원 규모와 모순되지 않는다"**까지다.
2. **"산단 때문에 주민 온열질환이 늘었다"** — 원 노트북 §6에서 산업활동과 주민 온열질환은 **무관**했다(작업장 노동으로 판별). 이 파이프라인이 그 결론을 뒤집지 않는다.
3. **β를 확정값처럼 사용** — 아래 두 편향이 **반대 방향**이라 순효과를 모른다.

### 불확실성의 구조
- **회귀희석** → β를 **과소**평가 (관측 LST에 오차가 있으면 기울기가 눌린다)
- **극단 지표로의 외삽 포화** → β를 **과대**평가 (관측소는 +17℃ 지표를 겪지 않는다)

### 시각에 대해 (원 노트북 §15-f)
Landsat 통과는 **10~11시 KST**로 일최고기온 시각(14~16시)과 다르다. 시간자료로 검정한 결과 이 차이는 **오차가 아니라 결합강도의 실제 일변화**였다 — β는 오전에서 오후로 갈수록 커지고(ASOS 10시 0.059 → 15시 0.180), **일최고 β ≈ 14~15시 β**다. 온열질환이 14~16시에 몰리므로 **일최고기온이 목적에 맞는 지표**다.

---

*원 노트북: `산업전력_검증노트북_2026-07-18.ipynb` §14·§15 · 산출물: `DERIVED_0720_산단_LST_반경별.csv`, `DERIVED_0720_관측소_LST기온_패널.csv`*